## 1. ライブラリのインポートと設定

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from pathlib import Path
import json
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, roc_auc_score, average_precision_score
import lightgbm as lgb
from scipy import sparse

# 日本語フォント設定（必要に応じて）
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# スタイル設定
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## 2. データの読み込み

分析結果ディレクトリを指定して、必要なファイルを読み込みます。

In [ ]:
# 分析結果ディレクトリを指定（最新のものを使用する場合は適宜変更）
result_dir = Path("results/experiments/feature_selection_prompt_last_token_20251129_214443")

# データディレクトリ
data_dir = result_dir / "data"

# メインデータの読み込み
feature_metrics = pd.read_csv(data_dir / "feature_metrics_full.csv")
candidates_suppress = pd.read_csv(data_dir / "candidates_suppress.csv")
candidates_amplify = pd.read_csv(data_dir / "candidates_amplify.csv")

print(f"全特徴数: {len(feature_metrics)}")
print(f"抑制候補数: {len(candidates_suppress)}")
print(f"増幅候補数: {len(candidates_amplify)}")
print("\n特徴メトリクスのカラム:")
print(feature_metrics.columns.tolist())

### データの概要確認

In [ ]:
# 基本統計量
print("=== 基本統計量 ===")
feature_metrics[['Freq Diff Base-Syc', 'Log Ratio Syc/Base', 'Diff Base-Syc', 'SHAP Correlation', 'Suppression Score', 'Amplification Score']].describe()

In [ ]:
# 上位抑制候補
print("\n=== 上位抑制候補（Top 5） ===")
candidates_suppress.head()

In [ ]:
feature_metrics[["Specificity", "Freq NonSyc (%)", "Mean Intensity Base", "Mean Intensity Syc","Amplification Score"]].describe()

In [ ]:
# 上位増幅候補
print("\n=== 上位増幅候補（Top 5） ===")
candidates_amplify[["Feature_ID","Specificity", "Freq NonSyc (%)", "Mean Intensity Base", "Mean Intensity Syc", "Suppression Score", "Amplification Score"]].head()

## 3. 特徴分布の可視化（Scatter Plot）

### 準備: 介入候補のマーキング

In [ ]:
# 介入候補のTop 20を取得
top_suppress_ids = set(candidates_suppress.head(20)['Feature_ID'].values)
top_amplify_ids = set(candidates_amplify.head(20)['Feature_ID'].values)

# マーカー用のカテゴリ列を追加
def categorize_feature(row):
    feature_id = row['Feature_ID']
    if feature_id in top_suppress_ids:
        return 'Suppress (Top 20)'
    elif feature_id in top_amplify_ids:
        return 'Amplify (Top 20)'
    else:
        return 'Other'

feature_metrics['Category'] = feature_metrics.apply(categorize_feature, axis=1)

# カテゴリごとの数を確認
print(feature_metrics['Category'].value_counts())

In [ ]:
# 日本語フォントを明示的に設定
plt.rcParams['font.sans-serif'] = ['Hiragino Sans', 'YuGothic', 'IPAexGothic', 'DejaVu Sans']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の文字化け対策

# スタイル設定
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

print("使用中のフォント:", plt.rcParams['font.sans-serif'][0])

In [ ]:
# フォントキャッシュをクリアして再構築
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

# フォントキャッシュを削除
fm._load_fontmanager(try_read_cache=False)

# 日本語フォントを設定
plt.rcParams['font.sans-serif'] = ['Hiragino Sans', 'YuGothic', 'IPAexGothic']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False

print("フォント設定完了")
print("現在のfont.sans-serif:", plt.rcParams['font.sans-serif'])
print("現在のfont.family:", plt.rcParams['font.family'])

### メイン散布図: Log Ratio vs Diff Base-Syc

In [ ]:
def plot_feature_distribution(df, top_suppress_ids, top_amplify_ids, figsize=(16, 10)):
    """
    特徴の分布を可視化する散布図を作成
    
    Parameters:
    -----------
    df : pd.DataFrame
        特徴メトリクスのデータフレーム
    top_suppress_ids : set
        抑制候補の特徴ID
    top_amplify_ids : set
        増幅候補の特徴ID
    figsize : tuple
        図のサイズ
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    # データを3つのグループに分割
    df_other = df[df['Category'] == 'Other']
    df_suppress = df[df['Category'] == 'Suppress (Top 20)']
    df_amplify = df[df['Category'] == 'Amplify (Top 20)']
    
    # その他の特徴（背景）
    scatter1 = ax.scatter(
        df_other['Log Ratio Syc/Base'],
        df_other['Diff Base-Syc'],
        c=df_other['SHAP Correlation'],
        cmap='RdBu_r',
        alpha=0.3,
        s=20,
        vmin=-1,
        vmax=1,
        label='Other features'
    )
    
    # 抑制候補（目立たせる）
    scatter2 = ax.scatter(
        df_suppress['Log Ratio Syc/Base'],
        df_suppress['Diff Base-Syc'],
        c=df_suppress['SHAP Correlation'],
        cmap='RdBu_r',
        alpha=0.9,
        s=150,
        marker='D',
        edgecolors='black',
        linewidths=2,
        vmin=-1,
        vmax=1,
        label='Suppress candidates (Top 20)'
    )
    
    # 増幅候補（目立たせる）
    scatter3 = ax.scatter(
        df_amplify['Log Ratio Syc/Base'],
        df_amplify['Diff Base-Syc'],
        c=df_amplify['SHAP Correlation'],
        cmap='RdBu_r',
        alpha=0.9,
        s=150,
        marker='s',
        edgecolors='green',
        linewidths=2,
        vmin=-1,
        vmax=1,
        label='Amplify candidates (Top 20)'
    )
    
    # カラーバー
    cbar = plt.colorbar(scatter1, ax=ax)
    cbar.set_label('SHAP Correlation', fontsize=12)
    
    # 軸ラベルとタイトル
    ax.set_xlabel('Log Ratio Syc/Base (対数倍率)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Diff Base-Syc (強度差分)', fontsize=12, fontweight='bold')
    ax.set_title('SAE Feature Distribution: Intervention Candidates Highlighting', 
                 fontsize=14, fontweight='bold', pad=20)
    
    # 凡例
    ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
    
    # グリッド
    ax.grid(True, alpha=0.3)
    
    # 参照線（x=0, y=0）
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    
    # 領域の説明を追加
    ax.text(0.98, 0.98, 'Suppress Zone\n(右上)', 
            transform=ax.transAxes, fontsize=10, 
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.text(0.02, 0.02, 'Amplify Zone\n(左下)', 
            transform=ax.transAxes, fontsize=10,
            verticalalignment='bottom', horizontalalignment='left',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
    plt.tight_layout()
    return fig, ax

# プロット実行
fig, ax = plot_feature_distribution(feature_metrics, top_suppress_ids, top_amplify_ids)
plt.show()

# 図を保存
# fig.savefig(result_dir / "figures" / "feature_distribution_scatter.png", dpi=300, bbox_inches='tight')
# print(f"図を保存しました: {result_dir / 'figures' / 'feature_distribution_scatter.png'}")

### 追加の可視化: SHAP Correlationの分布

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# ヒストグラム
axes[0].hist(feature_metrics['SHAP Correlation'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero line')
axes[0].set_xlabel('SHAP Correlation', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of SHAP Correlation', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Suppression Scoreの分布
axes[1].hist(feature_metrics['Suppression Score'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Suppression Score', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of Suppression Score', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Amplification Scoreの分布
axes[2].hist(feature_metrics['Amplification Score'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[2].set_xlabel('Amplification Score', fontsize=12)
axes[2].set_ylabel('Frequency', fontsize=12)
axes[2].set_title('Distribution of Amplification Score', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)


plt.tight_layout()
plt.show()

# 保存
# fig.savefig(result_dir / "figures" / "score_distributions.png", dpi=300, bbox_inches='tight')
# print(f"図を保存しました: {result_dir / 'figures' / 'score_distributions.png'}")

## 4. インタラクティブな特徴確認

特定の特徴IDの詳細情報を表示する関数

In [ ]:
def show_feature_details(feature_id, df):
    """
    指定された特徴IDの詳細情報を表示
    
    Parameters:
    -----------
    feature_id : int
        特徴ID
    df : pd.DataFrame
        特徴メトリクスのデータフレーム
    """
    feature = df[df['Feature_ID'] == feature_id]
    
    if len(feature) == 0:
        print(f"特徴ID {feature_id} は見つかりませんでした。")
        return
    
    feature = feature.iloc[0]
    
    print("=" * 60)
    print(f"特徴ID: {feature_id}")
    print("=" * 60)
    print(f"カテゴリ: {feature['Category']}")
    print("\n--- 基本指標 ---")
    print(f"Activation Count (Base): {feature['Freq Syc (%)']:.0f}")
    print(f"Activation Count (Syc): {feature['Freq NonSyc (%)']:.0f}")
    print(f"Mean Intensity (Base): {feature['Mean Intensity Syc']:.6f}")
    print(f"Mean Intensity (Syc): {feature['Mean Intensity NonSyc']:.6f}")
    print(f"Mean Intensity Index (Base): {feature['Mean Intensity Base']:.6f}")
    print("\n--- 重要な指標 ---")
    print(f"Log Ratio Syc/Base: {feature['Log Ratio Syc/Base']:.6f}")
    print(f"Diff Base-Syc: {feature['Diff Base-Syc']:.6f}")
    print(f"SHAP Correlation: {feature['SHAP Correlation']:.6f}")
    print(f"Specificity (Syc): {feature['Specificity']:.6f}")
    print(f"Suppression Score: {feature['Suppression Score']:.6f}")
    print(f"Amplification Score: {feature['Amplification Score']:.6f}")
    print("=" * 60)

# 使用例: 上位抑制候補の1番目を表示
if len(candidates_suppress) > 0:
    top_suppress_id = candidates_suppress.iloc[0]['Feature_ID']
    print("【抑制候補 Top 1】")
    show_feature_details(top_suppress_id, feature_metrics)

# 上位増幅候補の1番目を表示
if len(candidates_amplify) > 0:
    top_amplify_id = candidates_amplify.iloc[0]['Feature_ID']
    print("\n【増幅候補 Top 1】")
    show_feature_details(top_amplify_id, feature_metrics)

In [ ]:
# 任意の特徴IDを指定して詳細を確認
# 以下の数値を変更して実行してください
custom_feature_id = 12345  # 確認したい特徴IDを指定
show_feature_details(custom_feature_id, feature_metrics)

## 6. まとめ

このノートブックでは以下を実行しました:

1. **特徴分布の可視化**: Log Ratio vs Diff Base-Sycの散布図で、介入候補の位置を確認
2. **モデル性能評価**: ROC曲線、PR曲線、混同行列により、LightGBMモデルの性能を評価
3. **特徴詳細の確認**: 任意の特徴IDの詳細情報を表示する機能

### 次のステップ
- 特定された介入候補特徴を用いて、ステップ4の介入実験を実施
- 介入効果の評価と分析